# Notebook 10: is the resume-prior just fit on related jobs?

The resume-prior (a resume's mean label over its *other* jobs) predicts Good vs Potential
within a job (per-job AUC 0.727, Notebook 3b). If a resume's other jobs are in the same
occupation as this job, the prior can carry **real fit**, not a resume-level artifact.

**Test.** Recompute the prior using only the resume's other jobs that are **dissimilar** to the target job
(job-job cosine below the target job's median similarity to all other jobs), and see how much of the 0.727 survives.

**Rule (fixed in ANALYSIS-PLAN.md before this notebook was run):** primary = E5 job similarity, dissimilar half,
per-job Good-vs-Potential AUC on all candidates (prior falls back to the global mean when a resume has no dissimilar job,
exactly as Notebook 3b does when it has no other job).
- **KEPT**: CI lower bound > 0.5 **and** at least half of the full prior's excess over chance remains
  ((AUC − 0.5) / (0.727 − 0.5) ≥ 0.5).
- **PARTLY**: CI lower bound > 0.5 but less than half remains.
- **WITHDRAWN**: CI includes 0.5.

CPU runtime is enough (about 5–10 minutes). Run all cells in order.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install -U sentence-transformers

import os
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer

DATA = '/content/drive/MyDrive/resume-job-matching/data'
RESULTS = '/content/drive/MyDrive/resume-job-matching/results'
OUT = f'{RESULTS}/10_prior_dissimilar'
os.makedirs(OUT, exist_ok=True)
SEED = 42

all_df = pd.concat([pd.read_parquet(f'{DATA}/{s}.parquet').assign(split=s) for s in ('train', 'val', 'test')],
                   ignore_index=True)
g = all_df.groupby('job_id').agg(candidates=('resume_id', 'size'),
                                 relevant=('relevance', lambda r: int((r > 0).sum())),
                                 levels=('relevance', 'nunique'))
usable = g[(g.candidates >= 5) & (g.relevant >= 1) & (g.levels >= 2)].index
evald = all_df[all_df.job_id.isin(usable)]
jobs = {jid: grp.reset_index(drop=True) for jid, grp in evald.groupby('job_id')}
print(f'{len(jobs)} usable jobs, {len(evald)} pairs, {evald.resume_id.nunique()} resumes (Notebook 3b: 294 / 7692 / 641)')
assert len(jobs) == 294 and len(evald) == 7692

## 1. Reproduce the full resume-prior of Notebook 3b

In [ ]:
tot = all_df.groupby('resume_id').relevance.agg(['sum', 'count'])
global_mean = all_df.relevance.mean()

def gp_auc(rel, s):
    fit = rel > 0
    if fit.sum() < 2:
        return np.nan
    y = (rel[fit] == 2).astype(int)
    return roc_auc_score(y, s[fit]) if 0 < y.sum() < len(y) else np.nan

def fn_auc(rel, s):
    y = (rel > 0).astype(int)
    return roc_auc_score(y, s) if 0 < y.sum() < len(y) else np.nan

def per_job(scores):
    """scores: dict job_id -> array; NaN scores (uncovered candidates) are dropped for that job."""
    rows = []
    for jid, grp in jobs.items():
        rel = grp.relevance.to_numpy()
        s = np.asarray(scores[jid], dtype=float)
        keep = ~np.isnan(s)
        rows.append({'job_id': jid, 'gp': gp_auc(rel[keep], s[keep]), 'fn': fn_auc(rel[keep], s[keep])})
    return pd.DataFrame(rows).set_index('job_id')

full = {jid: np.array([(tot.loc[r, 'sum'] - rel) / (tot.loc[r, 'count'] - 1) if tot.loc[r, 'count'] > 1 else global_mean
                       for r, rel in zip(grp.resume_id, grp.relevance)])
        for jid, grp in jobs.items()}
pj_full = per_job(full)

ref = pd.read_csv(f'{RESULTS}/03b_all_jobs_per_query.csv')
ref = ref[ref.method == 'Resume-prior (no job text)'].set_index('job_id').reindex(pj_full.index)
for mine, theirs in [('gp', 'AUC good-vs-potential'), ('fn', 'AUC fit-vs-no')]:
    assert (pj_full[mine].isna() == ref[theirs].isna()).all(), f'{mine}: different jobs than Notebook 3b'
    diff = (pj_full[mine] - ref[theirs]).abs().max()
    print(f'{mine}: max difference from Notebook 3b = {diff:.2e}')
    assert diff < 1e-9, 'Does not reproduce Notebook 3b: stop here'
FULL_GP = pj_full.gp.mean()
print(f'Reproduced: per-job G/P AUC {FULL_GP:.3f} over {pj_full.gp.notna().sum()} mixed jobs; '
      f'fit/no {pj_full.fn.mean():.3f} over {pj_full.fn.notna().sum()} jobs')

## 2. How similar are jobs to each other? (E5 and TF-IDF, job texts only)

In [ ]:
from sentence_transformers import SentenceTransformer

job_text = all_df.drop_duplicates('job_id').set_index('job_id').job_text
JIDS = job_text.index.tolist()
pos = {j: i for i, j in enumerate(JIDS)}

e5 = SentenceTransformer('intfloat/e5-base-v2', device='cpu')
e5.max_seq_length = 512
E = e5.encode(['query: ' + t for t in job_text], batch_size=16, normalize_embeddings=True,
              convert_to_numpy=True, show_progress_bar=True)
S_E5 = E @ E.T

tv = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=100_000)
X = tv.fit_transform(job_text.tolist())
S_TF = (X @ X.T).toarray()
SIMS = {'E5': S_E5, 'TF-IDF': S_TF}

# premise check: are a resume's other jobs more similar to the target job than jobs in general?
res_jobs = {r: list(zip(d.job_id, d.relevance)) for r, d in all_df.groupby('resume_id')}
prem = []
for name, S in SIMS.items():
    own, base = [], []
    for jid, grp in jobs.items():
        row = S[pos[jid]]
        others_all = np.delete(row, pos[jid])
        for r in grp.resume_id:
            o = [row[pos[j2]] for j2, _ in res_jobs[r] if j2 != jid]
            if o:
                own.append(np.mean(o)); base.append(others_all.mean())
    prem.append({'similarity': name, 'pairs': len(own), "mean sim. to the resume's other jobs": round(np.mean(own), 3),
                 'mean sim. to all other jobs': round(np.mean(base), 3)})
premise = pd.DataFrame(prem)
premise

## 3. The prior over dissimilar (and, for contrast, similar) jobs only

In [ ]:
def make_prior(S, mode, q, fallback=True):
    scores, covered = {}, []
    for jid, grp in jobs.items():
        row = S[pos[jid]]
        cut = np.quantile(np.delete(row, pos[jid]), q)
        out = []
        for r in grp.resume_id:
            sel = [rel for j2, rel in res_jobs[r] if j2 != jid and
                   ((row[pos[j2]] < cut) if mode == 'dissimilar' else (row[pos[j2]] >= cut))]
            covered.append(len(sel) > 0)
            out.append(np.mean(sel) if sel else (global_mean if fallback else np.nan))
        scores[jid] = np.array(out, dtype=float)
    return scores, float(np.mean(covered))

VARIANTS = [  # (name, similarity, mode, quantile, fallback)
    ('E5, dissimilar half (PRIMARY)', 'E5', 'dissimilar', 0.50, True),
    ('E5, dissimilar half, covered candidates only', 'E5', 'dissimilar', 0.50, False),
    ('E5, least similar quarter', 'E5', 'dissimilar', 0.25, True),
    ('TF-IDF, dissimilar half', 'TF-IDF', 'dissimilar', 0.50, True),
    ('TF-IDF, least similar quarter', 'TF-IDF', 'dissimilar', 0.25, True),
    ('E5, similar half (contrast)', 'E5', 'similar', 0.50, True),
]
priors = {'Full prior (Notebook 3b)': (full, 1.0)}
for name, sim, mode, q, fb in VARIANTS:
    priors[name] = make_prior(SIMS[sim], mode, q, fb)
    print(f'{name:48} candidates with at least one such job: {priors[name][1]:.1%}')

## 4. Results and the pre-fixed rule

In [ ]:
def boot_ci(v, n=2000):
    v = np.asarray(v, dtype=float); v = v[~np.isnan(v)]
    rng = np.random.default_rng(SEED)
    return np.percentile([rng.choice(v, len(v), replace=True).mean() for _ in range(n)], [2.5, 97.5])

def signflip_p(d, n=10000):
    d = np.asarray(d, dtype=float); d = d[~np.isnan(d)]
    rng = np.random.default_rng(SEED)
    signs = rng.choice([-1, 1], size=(n, len(d)))
    return float((np.abs((signs * d).mean(axis=1)) >= abs(d.mean())).mean())

rows = []
for name, (sc, cov) in priors.items():
    pj = per_job(sc)
    for metric, label in [('gp', 'Good vs Potential (mixed jobs)'), ('fn', 'fit vs no fit')]:
        v = pj[metric]; lo, hi = boot_ci(v)
        both = pd.concat([v, pj_full[metric]], axis=1, keys=['v', 'f']).dropna()
        rows.append({'prior': name, 'task': label, 'jobs': int(v.notna().sum()), 'coverage': round(cov, 3),
                     'per-job AUC': round(v.mean(), 3), 'CI': f'[{lo:.3f}, {hi:.3f}]', 'CI low': lo,
                     'p vs 0.5': signflip_p(v - 0.5),
                     'diff vs full': round((both.v - both.f).mean(), 3) if name != 'Full prior (Notebook 3b)' else 0.0,
                     'p vs full': signflip_p(both.v - both.f) if name != 'Full prior (Notebook 3b)' else np.nan,
                     'share of excess kept': round((v.mean() - 0.5) / (FULL_GP - 0.5), 2) if metric == 'gp' else np.nan})
res = pd.DataFrame(rows)

prim = res[(res.prior == 'E5, dissimilar half (PRIMARY)') & (res.task.str.startswith('Good'))].iloc[0]
if prim['CI low'] > 0.5 and prim['share of excess kept'] >= 0.5:
    verdict = 'KEPT: the resume-prior is not explained by fit on related jobs'
elif prim['CI low'] > 0.5:
    verdict = 'PARTLY: part of the resume-prior is fit on related jobs'
else:
    verdict = 'WITHDRAWN: the resume-prior may be fit on related jobs'
print(f"PRIMARY: per-job G/P AUC {prim['per-job AUC']:.3f} {prim['CI']}, share of excess kept {prim['share of excess kept']:.2f}")
print('RULE ->', verdict)
res.drop(columns='CI low')

In [ ]:
res.drop(columns='CI low').to_csv(f'{OUT}/10_prior_dissimilar.csv', index=False)
premise.to_csv(f'{OUT}/10_premise_similarity.csv', index=False)
with open(f'{OUT}/10_verdict.txt', 'w') as f:
    f.write(verdict + '\n')
print('saved to', OUT)
print('Next: File > Download > Download .ipynb.')